# 04 Hybrid Inference + Heatmaps
Try the full pipeline on uploads:
- Faster R-CNN does backbone, proposals, boxes; the tree does labels.
- One panel per decision node — each one shows the region THAT step weighed, not one
  blended map. Read the panels as a trace ("first this region, then that one"), not a
  single localization claim.

In [ ]:
from pathlib import Path
import io

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import torch
import pandas as pd
from IPython.display import clear_output, display
from PIL import Image
from torch.utils.data import DataLoader

from notebooks.util import resolve_root
from neuro.config import NeuroConfig, NeuroTrainConfig
from neuro.prepare_dataset import PCBDataset
from neuro.preprocess_dataset import test_preprocess
from neuro.train import evaluate_model
from neuro.utils import detection_collate_fn
from neurosym.inference import (
    explain_hybrid_detection,
    load_neurosymbolic_detector,
    run_neurosymbolic_inference,
    select_detection_indices,
)
from neurosym.visualization import (
    draw_neurosymbolic_explanation,
    draw_numbered_detections,
)
from util.config import load_yaml
from util.device import select_device

from torchvision.ops import box_iou as _box_iou
from symbolic.evaluation import (
    evaluate_symbolic_model,
    evaluate_symbolic_spatial_metrics,
)
from neurosym.evaluation import (
    evaluate_faithfulness_fpn_masking,
    evaluate_exact_attribution_spatial_metrics,
)
from tqdm import tqdm as _tqdm

PROJECT_ROOT = resolve_root()

In [ ]:
neuro_config = load_yaml(Path("neuro.yaml"), NeuroConfig)
train_config = load_yaml(Path("neuro_train.yaml"), NeuroTrainConfig)

device = select_device(train_config["device"])
detector_checkpoint_path = PROJECT_ROOT / "checkpoints" / "neuro" / "run1.pt"
symbolic_checkpoint_path = PROJECT_ROOT / "checkpoints" / "symbolic" / "run1.pt"

hybrid_model, detector_checkpoint = load_neurosymbolic_detector(
    detector_checkpoint_path=detector_checkpoint_path,
    neuro_config=neuro_config,
    train_config=train_config,
    symbolic_checkpoint_path=symbolic_checkpoint_path,
    device=str(device),
)

image_preprocess = test_preprocess()
class_names = tuple(train_config["dataset"]["class_names"])

detector_checkpoint_path, symbolic_checkpoint_path

## Batch Evaluation

In [ ]:
test_dataset = PCBDataset(train_config, "test.txt")
test_dataset.add_preprocess(test_preprocess())
test_loader = DataLoader(
    test_dataset,
    batch_size=train_config["dataset"]["batch_size"],
    shuffle=False,
    collate_fn=detection_collate_fn,
)
print(f"Test samples: {len(test_dataset)}")

metrics = evaluate_model(
    model=hybrid_model,
    data_loader=test_loader,
    device=device,
    train_config=train_config,
)

skip_keys = {"precision_recall_score_threshold", "inference_time_ms", "parameter_count"}
metric_frame = pd.DataFrame(
    [
        {
            key: value
            for key, value in metrics.items()
            if isinstance(value, (int, float)) and key not in skip_keys
        }
    ]
)
ax = metric_frame.T.plot(kind="bar", legend=False, figsize=(9, 4), color="#2f6f73")
ax.set_title("NeSy Detection Metrics")
ax.set_ylabel("score")
ax.set_ylim(0.0, 1.05)
ax.grid(axis="y", alpha=0.25)
ax.bar_label(ax.containers[0], fmt="%.3f", padding=2, fontsize=8)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

if "per_class_AP" in metrics:
    s = pd.Series(metrics["per_class_AP"], name="AP").sort_values(ascending=False)
    ax = s.plot(kind="bar", figsize=(9, 4), color="#2f6f73")
    ax.set_title("NeSy Per-Class AP@[0.50:0.95]")
    ax.set_ylim(0.0, 1.05)
    ax.grid(axis="y", alpha=0.25)
    ax.bar_label(ax.containers[0], fmt="%.3f", padding=2, fontsize=8)
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

if "confusion_matrix" in metrics:
    cm = np.array(metrics["confusion_matrix"])
    labels = metrics["confusion_matrix_labels"]
    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(len(labels)))
    ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=9)
    ax.set_yticklabels(labels, fontsize=9)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title("NeSy Detection Confusion Matrix")
    cm_max = cm.max()
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            color = "white" if cm[i, j] > cm_max * 0.5 else "black"
            ax.text(
                j, i, str(cm[i, j]), ha="center", va="center", fontsize=7, color=color
            )
    fig.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.show()

if "per_class_precision" in metrics:
    pr = [k for k in metrics["per_class_precision"] if k != "__background__"]
    x = np.arange(len(pr))
    w = 0.35
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.bar(
        x - w / 2,
        [metrics["per_class_precision"][c] for c in pr],
        w,
        label="Precision",
        color="#2F6F9F",
    )
    ax.bar(
        x + w / 2,
        [metrics["per_class_recall"][c] for c in pr],
        w,
        label="Recall",
        color="#D97941",
    )
    ax.set_xticks(x)
    ax.set_xticklabels(pr, rotation=45, ha="right")
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Score")
    ax.set_title("NeSy Per-Class Precision and Recall")
    ax.legend()
    ax.grid(axis="y", alpha=0.25)
    ax.bar_label(ax.containers[0], fmt="%.3f", padding=2, fontsize=8)
    ax.bar_label(ax.containers[1], fmt="%.3f", padding=2, fontsize=8)
    plt.tight_layout()
    plt.show()
    print(f"Inference time: {metrics.get('inference_time_ms', 0):.2f} ms")
    print(f"Parameters: {metrics.get('parameter_count', 0):,}")

plt.close("all")
metric_frame

## Explanation Metrics
Two questions, two tables below: is the decomposition as a whole faithful (path-level, vs
GradCAM-style random baseline), and does each individual step's own region decide that
step's own question (per-node, SODT only — no other method has steps to compare against).

In [ ]:
tree = hybrid_model.symbolic_tree

all_features = []
all_grids = []
all_proposal_boxes = []
all_matched_gt_boxes = []
all_has_matched = []
all_gt_iou = []

for i in _tqdm(range(len(test_dataset)), desc="Collecting NeSy detections"):
    image, target = test_dataset[i]
    detection = run_neurosymbolic_inference(hybrid_model, [image])[0]
    n_det = detection["boxes"].shape[0]
    if n_det == 0:
        continue

    pooled = detection["pooled_features"]
    proposals = detection["proposal_boxes"]
    gt_boxes = target["boxes"]
    gt_labels = target["labels"]

    all_features.append(pooled.flatten(start_dim=1).numpy())
    all_grids.append(pooled.numpy())
    all_proposal_boxes.append(proposals)

    if gt_boxes.numel() > 0:
        overlaps = _box_iou(proposals, gt_boxes)
        matched_iou, matched_idx = overlaps.max(dim=1)
        matched_boxes = gt_boxes[matched_idx]
        has_match = matched_iou > 0
        all_matched_gt_boxes.append(matched_boxes)
        all_has_matched.append(has_match)
        all_gt_iou.append(matched_iou)
    else:
        all_matched_gt_boxes.append(torch.zeros((n_det, 4)))
        all_has_matched.append(torch.zeros(n_det, dtype=torch.bool))
        all_gt_iou.append(torch.zeros(n_det))

feature_matrix = np.concatenate(all_features, axis=0).astype(np.float32)
feature_grids = torch.from_numpy(np.concatenate(all_grids, axis=0))
all_proposals_t = torch.cat(all_proposal_boxes, dim=0)
all_matched_gt_t = torch.cat(all_matched_gt_boxes, dim=0)
all_has_matched_t = torch.cat(all_has_matched, dim=0)
all_gt_iou_t = torch.cat(all_gt_iou, dim=0)

nesy_pred = tree.predict(feature_matrix)

# Sufficiency stays on the pooled 7x7 grid — see the cell below for why it's
# not discriminative — plus a random-ranking control on the same budget.
# Necessity, deletion/insertion AUC, and pointing/IoU moved to the FPN-native
# cells below (leaf_only was dropped from reporting).
nesy_sufficiency = evaluate_symbolic_model(
    tree, feature_matrix, nesy_pred, class_names, compute_auc=False
)  # AUC unused below (FPN-native per-node AUC covers it instead) — skip the expensive pass
nesy_sufficiency_random = evaluate_symbolic_model(
    tree, feature_matrix, nesy_pred, class_names, ranking="random", compute_auc=False
)

print(f"Total NeSy detections: {feature_matrix.shape[0]:,}")
print(
    f"Sufficiency (own ranking): {nesy_sufficiency['sufficiency_prediction_preservation']:.4f}  "
    f"(random ranking: {nesy_sufficiency_random['sufficiency_prediction_preservation']:.4f})"
)

### Faithfulness at native FPN resolution: path-level + per-node
Masking on the pooled grid shrinks finer maps to 7x7 first — unfair to them. Below, necessity
masks the FPN map directly, then re-pools, so each map is judged at its own resolution
(`neurosym/evaluation.py::evaluate_faithfulness_fpn_masking`). Random control included
(raw numbers don't show chance).

- **Path-level**: masks the whole path's map, checks whether the final label flips. Validates
  the decomposition as a whole — this is what `exact 0.876` / `shuffled_w` ~0.65 /
  `activation_only` ~0.07 (dev refs, n=960) bound.
- **Per-node**: masks one node's own map, checks whether THAT node's own routing sign flips,
  plus a deletion/insertion AUC over that node's routing confidence. Answers "does the region
  this node weighs actually decide this node's question" — the claim the per-node heatmaps in
  the next section make. No Grad-CAM column here: Grad-CAM has no per-step structure to test.
- Sufficiency isn't discriminative here (random ~0.99) — reported, not relied on.
- The exact map's localization also reported (`evaluate_exact_attribution_spatial_metrics`) —
  a boundary check ("not random, roughly near the defect"), not a localization claim.

In [ ]:
import math

# The faithfulness protocol and the exact attribution both need a live forward pass
# through the backbone (the FPN map isn't part of any precomputed export), so this runs on
# a subsample of the test set rather than all 500 images. Per-node is ~2 orders of magnitude
# more re-pools than path-level alone — this subsample size assumes per_node=True; time
# images[:5] yourself before trusting it on a different depth/rankings/auc_steps.
FAITHFULNESS_FPN_SUBSAMPLE = 60

fpn_images = []
for i in range(FAITHFULNESS_FPN_SUBSAMPLE):
    image, _ = test_dataset[i]
    fpn_images.append(image)

fpn_faithfulness = evaluate_faithfulness_fpn_masking(hybrid_model, fpn_images)

fpn_targets = []
for i in range(FAITHFULNESS_FPN_SUBSAMPLE):
    _, target = test_dataset[i]
    fpn_targets.append(target)

exact_attr_spatial = evaluate_exact_attribution_spatial_metrics(
    hybrid_model, fpn_images, fpn_targets, min_proposal_iou=0.5
)

print("Necessity flip rate, path-level (higher = more faithful):")
print(f"{'ranking':12} {'nec_flip':>10} {'n':>6}")
print("-" * 30)
for name, r in fpn_faithfulness["path"].items():
    print(f"{name:12} {r['necessity_prediction_flip_rate']:10.4f} {r['evaluated_roi_count']:6d}")

print("\nPer-node — does the region this node weighs decide this node's own question?")
print(f"{'ranking':10} {'depth':>6} {'nec_flip':>10} {'ceiling':>9} {'support':>9} {'del_auc':>9} {'ins_auc':>9} {'n':>6}")
print("-" * 76)
ceiling = fpn_faithfulness["node_necessity_ceiling"]
for name, per_depth in fpn_faithfulness["node"].items():
    for depth, r in per_depth.items():
        # Ceiling is "exact"-specific (full-mask bound) — only meaningful
        # against the "exact" row, and only tight when support <= 0.5 (its
        # ranking then covers the node's whole nonzero support). "random"
        # has no "support" (no map of its own), shown as "—".
        support = f"{'—':>9}" if math.isnan(r['support_fraction']) else f"{r['support_fraction']:9.4f}"
        print(
            f"{name:10} {depth + 1:>6} {r['necessity_prediction_flip_rate']:10.4f} "
            f"{ceiling.get(depth, float('nan')):9.4f} {support} "
            f"{r['deletion_auc']:9.4f} {r['insertion_auc']:9.4f} {r['evaluated_roi_count']:6d}"
        )
print(
    "(ceiling: max flip rate 'exact' can reach if it masks the whole box — not a bound "
    "for 'random'. support: fraction of the box with any contribution — ceiling is only "
    "tight for 'exact' when this is <= 0.5, the masking budget.)"
)

sim = fpn_faithfulness["node_map_similarity"]
print(
    f"\nMean cosine similarity between consecutive nodes' maps: "
    f"{sim['mean_cosine_similarity_between_consecutive_nodes']:.4f} "
    f"(n={sim['evaluated_pair_count']:,}) — low means each step weighs a different region."
)

print(
    f"\nExact attribution localization (min_proposal_iou=0.5):\n"
    f"  IoU overlap:  {exact_attr_spatial['box_grounded_roi_overlap']:.4f}\n"
    f"  Pointing game: {exact_attr_spatial['pointing_score']:.4f}\n"
    f"  n: {exact_attr_spatial['evaluated_roi_count']:,}"
)

# Same saturation problem WHAT-I-DID Sec 6.4 found (GT covers most of the grid
# at this proposal tightness) — this subset is where pointing/IoU still move.
low = exact_attr_spatial["low_gt_coverage"]
print(
    f"\nLow-GT-coverage subset (GT < 50% of grid, n={low['evaluated_roi_count']:,}):\n"
    f"  IoU overlap:  {low['box_grounded_roi_overlap']:.4f}\n"
    f"  Pointing game: {low['pointing_score']:.4f}"
)

In [ ]:
# Path-level necessity comes from `fpn_faithfulness` (computed in the cell
# above) — not recomputed here.
sodt_necessity = fpn_faithfulness["path"]["exact"]["necessity_prediction_flip_rate"]
random_necessity = fpn_faithfulness["path"]["random"]["necessity_prediction_flip_rate"]

fig, ax = plt.subplots(figsize=(7, 5))
labels = ["Sufficiency\n(own ranking)", "Sufficiency\n(random)", "Necessity\n(exact)", "Necessity\n(random)"]
values = [
    nesy_sufficiency["sufficiency_prediction_preservation"],
    nesy_sufficiency_random["sufficiency_prediction_preservation"],
    sodt_necessity,
    random_necessity,
]
colors = ["#2F6F9F", "#B0B0B0", "#2F6F9F", "#B0B0B0"]
bars = ax.bar(range(len(labels)), values, color=colors, edgecolor="white")
ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=9, fontweight="bold")
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylim(0, 1.12)
ax.set_ylabel("Score")
ax.set_title("SODT Faithfulness vs Random (higher = better, both metrics)", fontsize=12, fontweight="bold")
ax.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()

print(
    "Sufficiency barely beats random — not discriminative on this tree (see the cell above).\n"
    "Necessity should clear random by a wide margin; deletion/insertion AUC and pointing/IoU\n"
    "are in the tables above/below (per-node AUC has no single-bar summary — see 'Per-node' table)."
)

In [ ]:
def class_name(label: int) -> str:
    return class_names[int(label) - 1]

In [ ]:
# Cleanup any stale figures / outputs from previous runs of this cell
import matplotlib.pyplot as plt

plt.close("all")
clear_output(wait=True)

upload_widget = widgets.FileUpload(
    accept="image/*",
    multiple=False,
    description="Upload Image",
)
MAX_DISPLAY_DETECTIONS = 15
DISPLAY_SCORE_THRESHOLD = 0.3
detection_output = widgets.Output()
explanation_output = widgets.Output()
state = {
    "image_name": None,
    "image_tensor": None,
    "detection": None,
    "selected_indices": [],
}


def uploaded_file_record():
    value = upload_widget.value
    if isinstance(value, tuple):
        return value[0] if value else None
    if isinstance(value, dict):
        if "content" in value:
            return value
        return next(iter(value.values())) if value else None
    return None


def uploaded_content_bytes(file_record) -> bytes:
    content = file_record["content"]
    return content.tobytes() if isinstance(content, memoryview) else bytes(content)


def render_detection(detection_index: int) -> None:
    image_tensor = state["image_tensor"]
    detection = state["detection"]
    selected_indices = state["selected_indices"]
    explanation = explain_hybrid_detection(
        hybrid_model,
        detection,
        detection_index=detection_index,
        image_shape=tuple(image_tensor.shape[-2:]),
    )
    with explanation_output:
        clear_output(wait=True)
        selected_number = selected_indices.index(detection_index) + 1
        draw_neurosymbolic_explanation(
            image_tensor,
            detection,
            detection_index,
            explanation,
            class_names,
            hybrid_model.symbolic_tree,
            selected_number=selected_number,
        )


def make_detection_button(display_number: int, detection_index: int) -> widgets.Button:
    detection = state["detection"]
    label = int(detection["labels"][detection_index])
    score = float(detection["scores"][detection_index])
    button = widgets.Button(
        description=f"#{display_number} {class_name(label)} {score:.2f}",
        layout=widgets.Layout(width="180px"),
    )
    button.on_click(lambda _: render_detection(detection_index))
    return button


_processing_upload = False


def run_uploaded_inference(change=None) -> None:
    global _processing_upload
    if _processing_upload:
        return
    if change is not None and not change.get("new"):
        return
    _processing_upload = True
    try:
        file_record = uploaded_file_record()

        with detection_output:
            clear_output(wait=True)
        explanation_output.clear_output(wait=True)
        plt.close("all")

        if file_record is None:
            with detection_output:
                print("Upload one PCB image first.")
            return

        image_name = file_record.get("name", "uploaded_image")
        pil_image = Image.open(io.BytesIO(uploaded_content_bytes(file_record))).convert(
            "RGB"
        )
        image_tensor = image_preprocess(pil_image)
        detection = run_neurosymbolic_inference(hybrid_model, [image_tensor])[0]
        selected_indices = select_detection_indices(
            detection,
            score_threshold=DISPLAY_SCORE_THRESHOLD,
            max_detections=MAX_DISPLAY_DETECTIONS,
        )

        state["image_name"] = image_name
        state["image_tensor"] = image_tensor
        state["detection"] = detection
        state["selected_indices"] = selected_indices

        with detection_output:
            fig, axis = plt.subplots(figsize=(8, 8))
            draw_numbered_detections(
                axis, image_tensor, detection, selected_indices, class_names
            )
            axis.set_title(f"Neuro-Symbolic detections: {image_name}")
            plt.show()
            plt.close(fig)

            if not selected_indices:
                print("No detections were returned by the model.")
                return

            buttons = [
                make_detection_button(display_number, detection_index)
                for display_number, detection_index in enumerate(
                    selected_indices, start=1
                )
            ]
            display(
                widgets.GridBox(
                    buttons,
                    layout=widgets.Layout(
                        grid_template_columns="repeat(3, 190px)",
                        grid_gap="8px",
                    ),
                )
            )

        if selected_indices:
            render_detection(selected_indices[0])
    finally:
        _processing_upload = False


upload_widget.observe(run_uploaded_inference, names="value")

ui = widgets.VBox(
    [
        upload_widget,
        detection_output,
        explanation_output,
    ]
)
display(ui)